# 20 - Shakkala (LSTM-based Arabic Diacritization)

**Model:** Shakkala (PyTorch port)
**Architecture:** B-LSTM + Character Embeddings
**Source:** github.com/nipponjo/arabic-vocalization
**Original:** github.com/Barqawiz/Shakkala
**Reported Performance:** ~95% accuracy, DER 2.88%

## About Shakkala
- Recurrent neural network for Arabic text vocalization
- Trained on over 1 million sentences (historical + modern Arabic)
- Uses B-LSTM with character embeddings

**Tasks:**
1. Dev Set: Inference + Metrics (DER, WER, SER)
2. Blind Test: Inference + Submission JSON

In [1]:
!pip install -q torch tqdm
!pip install -q git+https://github.com/nipponjo/arabic_vocalizer.git

In [2]:
# Setup
import os, sys, json, re, torch, zipfile, subprocess
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

try:
    from config import DEV_INPUT, DEV_OUTPUT, TEST_DIR, OUTPUT_DIR, SUBMISSION_DIR
    TEST_INPUT = TEST_DIR / 'test_input.json'
except ImportError:
    print("WARNING: config.py not found.")
    DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
    DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
    DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
    TEST_DIR = PROJECT_DIR / 'Test'
    TEST_INPUT = TEST_DIR / 'test_input.json'
    OUTPUT_DIR = PROJECT_DIR / 'outputs'
    SUBMISSION_DIR = PROJECT_DIR / 'submissions'

OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")

Environment: Local | Device: cuda
GPU: NVIDIA RTX A5000 (23.5 GB)


In [3]:
MODEL_KEY = 'shakkala'

# Load Shakkala model using arabic_vocalizer (ONNX-based)
from arabic_vocalizer import vocalize

# Test the model
test_text = "السلام عليكم"
test_result = vocalize(test_text, model='shakkala')
print(f"Shakkala model loaded successfully!")
print(f"Test: '{test_text}' -> '{test_result}'")

2026-02-22 05:20:45.838759074 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 9 Memcpy nodes are added to the graph tf2onnx for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.


Shakkala model loaded successfully!
Test: 'السلام عليكم' -> 'السَّلَامُ عَلَيْكُمْ'


In [4]:
def diacritize(text):
    """Diacritize Arabic text using Shakkala (ONNX)."""
    try:
        result = vocalize(text, model='shakkala')
        return result.strip() if result else text
    except Exception as e:
        print(f"Error: {e}")
        raise  # Don't silently fail - let the error propagate

In [5]:
# Metrics
DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')
ARABIC_LETTER_PATTERN = re.compile(r'[\u0621-\u064A]')
IRAB_DIACRITICS = {'\u064B', '\u064C', '\u064D'}

def extract_diacritics(text):
    result = []
    i = 0
    while i < len(text):
        if ARABIC_LETTER_PATTERN.match(text[i]):
            diacritics = []
            j = i + 1
            while j < len(text) and DIACRITIC_PATTERN.match(text[j]):
                diacritics.append(text[j])
                j += 1
            result.append((text[i], ''.join(diacritics)))
            i = j
        else:
            i += 1
    return result

def compute_metrics(predictions, ground_truth, exclude_irab=False):
    gt_lookup = {item['id']: item['text_diacritized'] for item in ground_truth}
    total_chars, total_words, char_errors, word_errors, ser_sum, n = 0, 0, 0, 0, 0, 0
    for pred in predictions:
        if pred['id'] not in gt_lookup: continue
        pred_text, ref_text = pred['text_diacritized'], gt_lookup[pred['id']]
        pred_pairs, ref_pairs = extract_diacritics(pred_text), extract_diacritics(ref_text)
        for i in range(min(len(pred_pairs), len(ref_pairs))):
            pred_d, ref_d = pred_pairs[i][1], ref_pairs[i][1]
            if exclude_irab:
                pred_d = ''.join(d for d in pred_d if d not in IRAB_DIACRITICS)
                ref_d = ''.join(d for d in ref_d if d not in IRAB_DIACRITICS)
            if pred_d != ref_d: char_errors += 1
        char_errors += abs(len(pred_pairs) - len(ref_pairs))
        total_chars += max(len(pred_pairs), len(ref_pairs))
        pred_words, ref_words = pred_text.split(), ref_text.split()
        for i in range(min(len(pred_words), len(ref_words))):
            if pred_words[i] != ref_words[i]: word_errors += 1
        word_errors += abs(len(pred_words) - len(ref_words))
        total_words += max(len(pred_words), len(ref_words))
        if pred_text != ref_text: ser_sum += 1
        n += 1
    return {'DER': char_errors/total_chars if total_chars else 0, 'WER': word_errors/total_words if total_words else 0, 'SER': ser_sum/n if n else 0, 'n_samples': n}

## Dev Set Inference

In [7]:
# Checkpoint support for resume
CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

with open(DEV_INPUT, 'r', encoding='utf-8') as f: dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f: dev_output = json.load(f)
print(f"Dev samples: {len(dev_input)}")

processed_ids, dev_predictions = load_checkpoint()
print(f"Resuming from checkpoint: {len(processed_ids)} already processed")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids:
        continue
    try:
        diacritized = diacritize(item['text_undiacritized'])
        dev_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
        save_checkpoint(dev_predictions)
    except Exception as e:
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

Dev samples: 260
Resuming from checkpoint: 0 already processed


Dev Set: 100%|██████████| 260/260 [00:25<00:00, 10.05it/s]


In [8]:
print("="*60 + "\nDEV SET RESULTS\n" + "="*60)
m1 = compute_metrics(dev_predictions, dev_output, False)
print(f"\n[With I'rab] DER: {m1['DER']*100:.2f}% | WER: {m1['WER']*100:.2f}% (PRIMARY) | SER: {m1['SER']*100:.2f}%")
m2 = compute_metrics(dev_predictions, dev_output, True)
print(f"[No I'rab]   DER: {m2['DER']*100:.2f}% | WER: {m2['WER']*100:.2f}% | SER: {m2['SER']*100:.2f}%")

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

DEV SET RESULTS

[With I'rab] DER: 21.81% | WER: 57.97% (PRIMARY) | SER: 96.92%
[No I'rab]   DER: 21.71% | WER: 57.97% | SER: 96.92%


## Blind Test Inference

In [9]:
# Test checkpoint
TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

with open(TEST_INPUT, 'r', encoding='utf-8') as f: test_input = json.load(f)
print(f"Test samples: {len(test_input)}")

test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Resuming from checkpoint: {len(test_processed_ids)} already processed")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids:
        continue
    try:
        diacritized = diacritize(item['text_undiacritized'])
        test_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
        save_test_checkpoint(test_predictions)
    except Exception as e:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)
zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print(f"\nSUBMISSION READY: {zip_file} ({zip_file.stat().st_size/1024:.1f} KB)")

Test samples: 328
Resuming from checkpoint: 0 already processed


Test Set: 100%|██████████| 328/328 [00:31<00:00, 10.49it/s]


SUBMISSION READY: ./submissions/shakkala_submission_20260222_052355.zip (18.9 KB)


In [10]:
# Google Drive sync removed for public release
